In [ ]:
try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

# caminho de entrada dos dados no databricks
dbutils.widgets.text("input_base_path", "/Volumes/workspace/default/inputs", "Pasta de entrada")
input_base_path = dbutils.widgets.get("input_base_path").rstrip("/")
if not input_base_path:
    raise ValueError("Informe o caminho da pasta de entrada no widget do Databricks.")

# lista os arquivos e os nomes das tabelas bronze
csv_sources = {
    "tb_movies_info": "movies_info_TMDB_IMDB.csv",
    "tb_movies_financials": "movies_financials_IMDB_TMDB.csv",
    "tb_movies_metrics": "movies_metrics_IMDB_TMDB.csv",
    "tb_credits_and_tags": "credits_and_tags_IMDB_TMDB.csv",
    "tb_movies_reviews": "movies_reviews.csv",
}

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [ ]:
# mantém as colunas como texto
csv_read_options = {
    "header": "true",
    "inferSchema": "false",
    "quote": '"',
    "escape": '"',
    "mode": "PERMISSIVE",
}

ingestion_timestamp = F.current_timestamp()

for table_name, file_name in csv_sources.items():
    source_path = f"{input_base_path}/{file_name}"

    (
        spark.read
        .options(**csv_read_options)
        .csv(source_path)
        # adição do horário da carga
        .withColumn("ingestion_datetime", ingestion_timestamp)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{table_name}")
    )

print(f"Carga concluída: {len(csv_sources)} tabelas foram gravadas na camada Bronze.")

In [ ]:
# verifica o formato das tabelas e a coluna de horário de ingestão dos dados
validation_dfs = []
for table_name in csv_sources:
    table_df = spark.table(f"bronze.{table_name}")
    detail_df = (
        spark.sql(f"DESCRIBE DETAIL bronze.{table_name}")
        .select(
            F.lit(table_name).alias("table_name"),
            F.lit("ingestion_datetime" in table_df.columns).alias("has_ingestion_datetime"),
            F.col("format").alias("storage_format"),
        )
    )
    validation_dfs.append(detail_df)

validation_result = validation_dfs[0]
for validation_df in validation_dfs[1:]:
    validation_result = validation_result.unionByName(validation_df)

display(validation_result)

In [ ]:
from datetime import date, datetime, timedelta
import requests
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# define o período da consulta e permite repetir a carga para qualquer intervalo
default_data_fim = date.today()
default_data_inicio = default_data_fim - timedelta(days=6)
dbutils.widgets.text("data_inicio", default_data_inicio.strftime("%d/%m/%Y"), "Data inicial (dd/MM/yyyy)")
dbutils.widgets.text("data_fim", default_data_fim.strftime("%d/%m/%Y"), "Data final (dd/MM/yyyy)")

data_inicio = datetime.strptime(dbutils.widgets.get("data_inicio"), "%d/%m/%Y").date()
data_fim = datetime.strptime(dbutils.widgets.get("data_fim"), "%d/%m/%Y").date()
if data_inicio > data_fim:
    raise ValueError("A data inicial deve ser menor ou igual à data final.")

print(f"Período da cotação: {data_inicio:%d/%m/%Y} a {data_fim:%d/%m/%Y}")

In [ ]:
# consulta a API PTAX com o endpoint e as datas exigidos pelo Banco Central
api_url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)
request_params = {
    "@dataInicial": f"'{data_inicio:%m-%d-%Y}'",
    "@dataFinalCotacao": f"'{data_fim:%m-%d-%Y}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
}
response = requests.get(api_url, params=request_params, timeout=30)
if response.status_code != 200:
    raise RuntimeError(f"Erro ao consultar a API do Banco Central: {response.status_code} - {response.text}")

# mantém somente os campos da cotação necessários para a camada Bronze
cotacoes = response.json().get("value", [])
cotacao_schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])
df_cotacao_dolar = (
    spark.createDataFrame(cotacoes, schema=cotacao_schema)
    .withColumn("ingestion_datetime", F.current_timestamp())
)

(
    df_cotacao_dolar.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

print(f"Coleta de dados concluída: {df_cotacao_dolar.count()} cotações gravadas em bronze.tb_cotacao_dolar.")

In [ ]:
# valida o formato Delta e a coluna de horário de ingestão da API
cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")
validation_cotacao = (
    spark.sql("DESCRIBE DETAIL bronze.tb_cotacao_dolar")
    .select(
        F.lit("tb_cotacao_dolar").alias("table_name"),
        F.lit("ingestion_datetime" in cotacao_bronze.columns).alias("has_ingestion_datetime"),
        F.lit("dataHoraCotacao" in cotacao_bronze.columns).alias("has_dataHoraCotacao"),
        F.lit("cotacaoCompra" in cotacao_bronze.columns).alias("has_cotacaoCompra"),
        F.col("format").alias("storage_format"),
    )
)

display(validation_cotacao)